# Inference – Pose from Cylinders (V2)

This notebook runs inference with the larger pair-geometry model in `model_v2.py`.

The original `model.py` remains unchanged for the existing small model / notebook.

The V2 model uses:

- shared patch embedding for both images
- per-image self-attention
- bidirectional cross-attention between the image streams
- camera tokens + register tokens
- learned cylinder query tokens for the `(occupancy, radius, depth)` output
- a pose head operating on the two camera tokens

The notebook keeps the same practical outputs as the old inference notebook: vision curves, cylinder overlays, relative pose, and optional training history.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset import SceneTwoPairsDataset
from model_v2 import PairImageCylinderModelV2
from utils import (
    plot_estimated_cylinders_on_images,
    plot_relative_pose,
    pose_to_text,
    show_image_pair,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Point this to the checkpoint produced by the new train.py.
CHECKPOINT = Path("arrhenius_runs/v2/checkpoints/latest.pt")
TEST_DATASET = Path("testdataset")

BATCH_SIZE = 4
SAMPLE_IDX = 0
OCC_THRESHOLD = 0.5

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("Checkpoint:", CHECKPOINT)
print("Test dataset:", TEST_DATASET)


In [ ]:
checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

args = checkpoint["args"]
print("Checkpoint epoch:", checkpoint["epoch"])
print("Model arguments:")
print(json.dumps(args, indent=2))

# V2-specific architectural arguments are read from the checkpoint when
# available, with the same defaults used by model_v2.py / train.py.
model = PairImageCylinderModelV2(
    img_size=args["img_size"],
    patch_size=args["patch_size"],
    in_chans=3,
    embed_dim=args["embed_dim"],
    depth=args["depth"],
    num_heads=args["num_heads"],
    num_bins=args["num_bins"],
    mlp_ratio=args.get("mlp_ratio", 4.0),
    num_register_tokens=args.get("num_register_tokens", 4),
    cylinder_decoder_depth=args.get("cylinder_decoder_depth", 2),
    dropout=args.get("dropout", 0.0),
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded successfully. Parameters: {n_params / 1e6:.2f} M")


In [ ]:
infer_dataset = SceneTwoPairsDataset(
    root_dir=str(TEST_DATASET),
    image_size=args["img_size"],
    return_two_pairs=False,
)

infer_loader = DataLoader(
    infer_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

print("Inference dataset size:", len(infer_dataset))


In [ ]:
batch = next(iter(infer_loader))
img_a, vision_a, img_b, vision_b, pose_ab = batch

img_a = img_a.to(DEVICE)
vision_a = vision_a.to(DEVICE)
img_b = img_b.to(DEVICE)
vision_b = vision_b.to(DEVICE)
pose_ab = pose_ab.to(DEVICE)

with torch.no_grad():
    pred_vision, pred_vision_b, pred_pose = model(img_a, img_b)

print("pred_vision shape:", pred_vision.shape)
print("pred_vision_b shape:", pred_vision_b.shape)
print("pred_pose shape:", pred_pose.shape)
print("GT pose:", pose_ab[SAMPLE_IDX].detach().cpu())
print("Pred pose:", pred_pose[SAMPLE_IDX].detach().cpu())
print("GT pose text:", pose_to_text(pose_ab[SAMPLE_IDX].detach().cpu()))
print("Pred pose text:", pose_to_text(pred_pose[SAMPLE_IDX].detach().cpu()))


## 1. Input image pair


In [ ]:
show_image_pair(
    img_a[SAMPLE_IDX],
    img_b[SAMPLE_IDX],
    title=f"Inference sample {SAMPLE_IDX}",
)


## 2. Vision prediction

Compare the predicted `(occupancy, radius, depth)` against the ground truth for image A and image B.


In [ ]:
sample_idx = SAMPLE_IDX
bins = np.arange(pred_vision.shape[1])

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
names = ["occupancy", "radius", "depth"]

for j, name in enumerate(names):
    ax = axes[j]
    gt_a = vision_a[sample_idx, :, j].detach().cpu().numpy()
    pr_a = pred_vision[sample_idx, :, j].detach().cpu().numpy()
    gt_b = vision_b[sample_idx, :, j].detach().cpu().numpy()
    pr_b = pred_vision_b[sample_idx, :, j].detach().cpu().numpy()

    ax.plot(bins, gt_a, label="GT A")
    ax.plot(bins, pr_a, "--", label="Pred A")
    ax.plot(bins, gt_b, label="GT B", alpha=0.65)
    ax.plot(bins, pr_b, "--", label="Pred B", alpha=0.65)
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("Bin")
fig.suptitle(f"Vision prediction sample {sample_idx}")
plt.tight_layout()
plt.show()


## 3. Estimated cylinders on the image pair


In [ ]:
plot_estimated_cylinders_on_images(
    [img_a[SAMPLE_IDX], img_b[SAMPLE_IDX]],
    [pred_vision[SAMPLE_IDX], pred_vision_b[SAMPLE_IDX]],
    titles=["Image A – predicted cylinders", "Image B – predicted cylinders"],
    occ_threshold=OCC_THRESHOLD,
    flip_x=True,
)


## 4. Relative pose


In [ ]:
gt_pose = pose_ab[SAMPLE_IDX].detach().cpu()
pred_pose_sample = pred_pose[SAMPLE_IDX].detach().cpu()

print("GT :", pose_to_text(gt_pose))
print("Pred:", pose_to_text(pred_pose_sample))

plot_relative_pose(
    gt_pose,
    pred_pose_sample,
)


## 5. Training history (optional)

If `history.json` exists next to the checkpoint directory, plot the available training curves.


In [ ]:
HISTORY = CHECKPOINT.parent.parent / "history.json"

if HISTORY.exists():
    with open(HISTORY, "r", encoding="utf-8") as f:
        history = json.load(f)

    if isinstance(history, list) and history:
        epochs = [row.get("epoch", i + 1) for i, row in enumerate(history)]
        keys = [k for k in history[0].keys() if k != "epoch"]

        fig, ax = plt.subplots(figsize=(10, 5))
        for key in keys:
            values = [row.get(key, np.nan) for row in history]
            if any(v is not None for v in values):
                ax.plot(epochs, values, label=key)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Value")
        ax.set_title("Training history")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("History file is empty or has an unexpected format:", HISTORY)
else:
    print("No history.json found at:", HISTORY)
